# NOTEBOOK : 06_train_models.ipynb
# ROLE     : Train 5 classification/ensemble models strictly on train.csv and save artifacts.
# INPUT    : ../../data/processed/train.csv, ../artifacts/features_list.json, ../artifacts/models/match_scaler.joblib
# OUTPUT   : ../artifacts/models/{logistic_regression,decision_tree,random_forest,gradient_boosting,xgboost}.joblib

In [1]:
from pathlib import Path
import json
import joblib
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

PROCESSED_DIR = Path("../../data/processed")
ARTIFACTS_DIR = Path("../artifacts")
MODELS_DIR = ARTIFACTS_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
with open(ARTIFACTS_DIR / "features_list.json", "r", encoding="utf-8") as f:
    FEATURE_COLS = json.load(f)

scaler = joblib.load(MODELS_DIR / "match_scaler.joblib")

X_train = train_df[FEATURE_COLS]
X_train_scaled = scaler.transform(X_train)
y_train = train_df["target"]  # 0=A, 1=D, 2=H

print(f"Training set: {X_train.shape[0]} matches, {X_train.shape[1]} features")
print("Target distribution:\n", y_train.value_counts().sort_index())

Training set: 1872 matches, 40 features
Target distribution:
 target
0    629
1    431
2    812
Name: count, dtype: int64


In [2]:
models = {
    "logistic_regression": (
        LogisticRegression(max_iter=1000, random_state=42),
        True,  # requires scaled features
    ),
    "decision_tree": (
        DecisionTreeClassifier(criterion="gini", max_depth=5, random_state=42),
        False,
    ),
    "random_forest": (
        RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
        False,
    ),
    "gradient_boosting": (
        GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, random_state=42),
        False,
    ),
    "xgboost": (
        XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            n_estimators=150,
            learning_rate=0.05,
            max_depth=3,
            eval_metric="mlogloss",
            random_state=42,
            n_jobs=-1,
        ),
        False,
    ),
}

for name, (model, use_scaled) in models.items():
    X_in = X_train_scaled if use_scaled else X_train
    model.fit(X_in, y_train)
    out_path = MODELS_DIR / f"{name}.joblib"
    joblib.dump(model, out_path)
    print(f"[SAVED] {name:20s} -> {out_path}")

[SAVED] logistic_regression  -> ..\artifacts\models\logistic_regression.joblib
[SAVED] decision_tree        -> ..\artifacts\models\decision_tree.joblib
[SAVED] random_forest        -> ..\artifacts\models\random_forest.joblib
[SAVED] gradient_boosting    -> ..\artifacts\models\gradient_boosting.joblib
[SAVED] xgboost              -> ..\artifacts\models\xgboost.joblib


In [3]:
for name, (model, use_scaled) in models.items():
    X_in = X_train_scaled if use_scaled else X_train
    probs = model.predict_proba(X_in)
    assert probs.shape == (len(y_train), 3), f"Unexpected probability shape for {name}"
    assert abs(probs.sum(axis=1).mean() - 1.0) < 1e-6, f"Probabilities do not sum to 1 for {name}"
print("All 5 models trained and verified successfully.")

All 5 models trained and verified successfully.
